In [1]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

E0000 00:00:1752279005.514294    2798 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752279005.518053    2798 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752279005.529236    2798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752279005.529258    2798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752279005.529261    2798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752279005.529263    2798 computation_placer.cc:177] computation placer already registered. Please check linka

# Check GPU Status / Availability

On linux systems you can and should use the `nvidia-smi` tool to check that the gpu is visible, is active and has drivers installed.  You can run the command from a terminal like the following cell.

I also find the following commands useful to monitor the gpu performance from the command line

```
# use watch so basic nvidia-smi redraws at top of screen each second
$ watch -n 1 nvidia-smi

# nvtop is basiclly like top for nvidia gpu
$ sudo apt install nvtop
$ nvtop

# nvitop is similar, gives about same information, but some may prefer this one
$ sudo apt install nvitop
$ nvitop
```

In [2]:
!nvidia-smi

Sat Jul 12 00:10:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.07             Driver Version: 570.133.07     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060        Off |   00000000:01:00.0  On |                  N/A |
|  0%   54C    P8             16W /  170W |    2008MiB /  12288MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# GPU to TF/Keras Availability

We can check that tensorflow recognizes the presence of a GPU device as follows.

In [3]:
print('Available Devices : ', tf.config.list_physical_devices())
print('Num GPUs Available: ', len(tf.config.list_physical_devices('GPU')))

Available Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Num GPUs Available:  1


# Test Training Performance with only CPU

We can compare performance with and without the gpu.  First we will train a model using only the cpu and record time it takes to train model.  We just train a simple dens NN on mnist data to test 
performance on fitting a model.

In [4]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0


loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

with tf.device('/CPU:0'):
    model_cpu_base = tf.keras.models.Sequential([
      tf.keras.layers.Input(shape=(28, 28)),
      tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(4096, activation='relu'),
      tf.keras.layers.Dropout(0.25),
      tf.keras.layers.Dense(10)
    ])


I0000 00:00:1752279035.705486    2798 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8178 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [5]:
def fit_cpu():
    with tf.device('/CPU:0'):
        model_cpu = tf.keras.models.clone_model(model_cpu_base)
        model_cpu.compile(optimizer='adam',
                      loss=loss_fn,
                      metrics=['accuracy'])
        model_cpu.fit(x_train, y_train, epochs=5)
        print('=' * 70)
        
%timeit fit_cpu()

Epoch 1/5


I0000 00:00:1752279036.472034    2951 service.cc:152] XLA service 0x7adcb8021b90 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1752279036.472058    2951 service.cc:160]   StreamExecutor device (0): Host, Default Version


  14/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.5073 - loss: 1.6476

I0000 00:00:1752279036.961284    2951 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9065 - loss: 0.3042
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9703 - loss: 0.0962
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9787 - loss: 0.0654
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9831 - loss: 0.0541
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9847 - loss: 0.0470
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9047 - loss: 0.3052
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9704 - loss: 0.0960
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9783 - loss: 0.0662
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9825 - loss: 0.0551
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9862 - loss: 0.0458
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9036 - loss: 0.3084
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1

# Test Training Performance with GPU

Now do same model and fit it on the gpu device.

In [6]:
# recreate a random starting model and compile it
with tf.device('/GPU:0'):
    model_gpu_base = tf.keras.models.Sequential([
      tf.keras.layers.Input(shape=(28, 28)),
      tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(4096, activation='relu'),
      tf.keras.layers.Dropout(0.25),
      tf.keras.layers.Dense(10)
    ])


In [7]:
def fit_gpu():
    with tf.device('/GPU:0'):
        model_gpu = tf.keras.models.clone_model(model_gpu_base)
        model_gpu.compile(optimizer='adam',
                      loss=loss_fn,
                      metrics=['accuracy'])
        model_gpu.fit(x_train, y_train, epochs=5)
        print('=' * 70)

%timeit fit_gpu()

Epoch 1/5


I0000 00:00:1752279625.868805    2952 service.cc:152] XLA service 0x7adcbc8baf20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1752279625.868824    2952 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
I0000 00:00:1752279625.938944    2952 cuda_dnn.cc:529] Loaded cuDNN version 90300


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9044 - loss: 0.3082
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9711 - loss: 0.0923
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9789 - loss: 0.0670
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9844 - loss: 0.0497
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9869 - loss: 0.0423
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9040 - loss: 0.3056
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9685 - loss: 0.1016
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9801 - loss: 0.0634
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9823 - loss: 0.0567
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9855 - loss: 0.0469
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9058 - loss: 0.3048
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step 